# 05_overfitting_and_generalization

Analyze overfitting and generalization with prediction capacity, perplexity, accuracy/error, and train-validation gaps — not validation loss alone. Smoke outputs are invalid for science.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wwgpt.analysis import (
    add_generalization_measures,
    completed_runs,
    discover_experiment_runs,
    summary,
    vocab_size_from_artifacts,
)
RESULTS_ROOT = Path('../results')
print('configured', RESULTS_ROOT)


In [ ]:
if RESULTS_ROOT.exists():
    runs = discover_experiment_runs(RESULTS_ROOT)
    completed = completed_runs(RESULTS_ROOT)
    print(f'discovered optimizer arms: {len(runs)}')
    print(f'completed scientific runs: {len(completed)}')
else:
    runs = []
    print('Set RESULTS_ROOT to an existing append-only results directory.')


## Generalization metrics

This section derives complementary measures from each run's metrics: validation perplexity, bits per token, token-prediction capacity relative to a uniform vocabulary baseline, top-k accuracy/error when present, and train-validation gaps.


In [ ]:
records = []
metric_frames = []
for run in runs:
    run_dir = run.get('run_dir')
    artifacts = run.get('artifacts', {})
    metrics = artifacts.get('metrics.csv')
    if run_dir is None or metrics is None or metrics.empty:
        continue
    vocab_size = vocab_size_from_artifacts(artifacts)
    m = add_generalization_measures(metrics, vocab_size=vocab_size)
    m['pair_id'] = run['pair_id']
    m['seed'] = run['seed']
    m['optimizer'] = run['optimizer']
    m['run_dir'] = str(run_dir)
    m['vocab_size'] = vocab_size
    metric_frames.append(m)

all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
if all_metrics.empty:
    print('No metrics.csv files found under RESULTS_ROOT.')
else:
    display(all_metrics.tail())


In [ ]:
generalization_cols = [
    'val_loss', 'val_perplexity', 'val_bits_per_token',
    'val_token_prediction_capacity', 'val_top1_accuracy', 'val_top5_accuracy',
    'val_token_error', 'generalization_gap', 'perplexity_gap',
    'perplexity_ratio', 'capacity_generalization_gap',
]
if not all_metrics.empty:
    sort_col = 'tokens_seen' if 'tokens_seen' in all_metrics.columns else 'step'
    final_metrics = all_metrics.sort_values(sort_col).groupby('run_dir', as_index=False).tail(1)
    available = [c for c in generalization_cols if c in final_metrics.columns]
    summary_rows = []
    for (optimizer,), group in final_metrics.groupby(['optimizer']):
        for metric in available:
            row = {'optimizer': optimizer, 'metric': metric}
            row.update(summary(group[metric]))
            summary_rows.append(row)
    generalization_summary = pd.DataFrame(summary_rows)
    display(generalization_summary)
else:
    final_metrics = pd.DataFrame()
    generalization_summary = pd.DataFrame()


## Paired optimizer differences

For paired seeds, differences are WW-PGD minus AdamW. Negative is better for loss/perplexity/error/gaps; positive is better for accuracy and token-prediction capacity.


In [ ]:
if not final_metrics.empty:
    paired_rows = []
    for metric in [c for c in generalization_cols if c in final_metrics.columns]:
        pivot = final_metrics.pivot_table(index=['pair_id', 'seed'], columns='optimizer', values=metric, aggfunc='first')
        if {'adamw', 'adamw_wwpgd'}.issubset(pivot.columns):
            diff = (pivot['adamw_wwpgd'] - pivot['adamw']).dropna()
            if len(diff):
                row = {'metric': metric, 'complete_pairs': len(diff), 'wwpgd_minus_adamw_mean': diff.mean()}
                row.update({f'diff_{k}': v for k, v in summary(diff).items()})
                paired_rows.append(row)
    paired_generalization = pd.DataFrame(paired_rows)
    display(paired_generalization)
else:
    paired_generalization = pd.DataFrame()


In [ ]:
if not all_metrics.empty:
    plot_metrics = [c for c in ['val_perplexity', 'val_token_prediction_capacity', 'val_top1_accuracy', 'generalization_gap'] if c in all_metrics.columns]
    sort_col = 'tokens_seen' if 'tokens_seen' in all_metrics.columns else 'step'
    fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(8, 3 * max(1, len(plot_metrics))), sharex=True)
    if len(plot_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, plot_metrics):
        for optimizer, group in all_metrics.groupby('optimizer'):
            curve = group.groupby(sort_col)[metric].mean()
            ax.plot(curve.index, curve.values, label=optimizer)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
    axes[-1].set_xlabel(sort_col)
    axes[0].legend()
    plt.tight_layout()
